In [1]:
import pysam
import numpy as np
import pandas as pd
import itertools
import re

from Bio import SeqIO
from Bio.Seq import Seq
from tqdm import tqdm
from collections import defaultdict

In [2]:
def get_align_flag(align, mapq_thre =20):
    is_paired  = align.is_proper_pair
    is_mapped  = (not align.is_unmapped)
    is_primary = (not align.is_secondary) and (not align.is_supplementary)
    is_qualified = (not align.is_qcfail) and align.mapq >= mapq_thre
    is_fully_align = all(op in (0, 7, 8) for op, length in align.cigar)
    return is_paired and is_mapped and is_primary and is_qualified and is_fully_align


def read_pair_generator(bam_file, region_string=None):
    """
    Generate read pairs in a BAM file or within a region string.
    Reads are added to read_dict until a pair is found.
    """
    read_dict = defaultdict(lambda: [None, None])
    bam = pysam.AlignmentFile(bam_file, "rb")
    #bam.reset()
    
    for align in bam.fetch(until_eof=True, region=region_string):        
        if not get_align_flag(align):
            continue
        
        qname = align.query_name
        if qname not in read_dict.keys():
            read_dict[qname][int(align.is_read2)] = align
        else:
            read_dict[qname][int(align.is_read2)] = align
            yield read_dict[qname]
            del read_dict[qname]

In [3]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + "/data/ref/GRCh38_full_analysis_set_plus_decoy_hla.fa"
bam_file  = working_path + "/data/TBS/20000.sorted.mdup.bam"

In [4]:
ref_dict  = SeqIO.to_dict(SeqIO.parse(ref_fasta, "fasta"))

In [5]:
read_len  = 151
read1_err = np.zeros((2, read_len)) # the first row record the total, second record the mistake
read2_err = np.zeros((2, read_len)) 

In [6]:
read1_err

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0.,

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [45]:
read2_err

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0.,

In [ ]:
# specifically for BSBolt, as it did not have the reverse tag...
for read1, read2 in read_pair_generator(bam_file, "chr21"):
    # check if the 2 reads overlap
    s1 = read1.reference_start
    e1 = read1.reference_end
    s2 = read2.reference_start
    e2 = read2.reference_end
    
    if read1.reference_name != read2.reference_name:
        continue
    
    if min(e2-s1, e1-s2) < 0:#s1==e1==s2==e2 or s2==e2==s1==e1
        continue
    
    if s1 > s2: # s2==s1==e2==e1 or s2==s1==e1==e2
        start = s1
        end = min(e1, e2)
        
        # make sure read1 and read2 both aligned to the same contig
        read1_pos = (0, end-start)
        read2_pos = (s1-s2, s1-s2+end-start)
        error_arr = check_overlap_err(read1, read1_pos, read2, read2_pos)
        read1_err += error_arr[0]
        read2_err += error_arr[1]
    else:       # s1==s2==e1==e2 or s1==s2==e2==e1
        start = s2
        end = min(e1, e2)
        read1_pos = (s2-s1, s2-s1+end-start)
        read2_pos = (0, end-start)
        error_arr = check_overlap_err(read1, read1_pos, read2, read2_pos)
        read1_err += error_arr[0]
        read2_err += error_arr[1]

In [50]:
read1.seq

'GGGAGTGGTGGGGGTGTTGGGGGTTGAGTGTGGAGGAGTTGGGAGTGGTGGGGGAGTTGGGGGTTGATTGTGGAGGAGTTGGGAGTGGTGGGGGAGTTGGGGGTTGAGTGTGGAGGAG'

In [44]:
read1.mate_is_reverse

False

In [43]:
read2.mate_is_reverse

False

In [28]:
my_string = 'hello world'
my_string2 = 'hello!eorld'

my_array = np.array(list(my_string))
my_array2= np.array(list(my_string2))

In [29]:
my_array

array(['h', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd'], dtype='<U1')

In [30]:
my_array2

array(['h', 'e', 'l', 'l', 'o', '!', 'e', 'o', 'r', 'l', 'd'], dtype='<U1')

In [31]:
idx = np.squeeze(np.where(my_array != my_array2))
idx

array([5, 6])

In [56]:
x = (1,6)

In [61]:
np.arange(x[0],x[1]) 

array([1, 2, 3, 4, 5])

In [ ]:
def get_seq_err(read1, read1_pos, read2, read2_pos):
    count_arr1 = np.zeros((2,read_len))
    count_arr2 = np.zeros((2,read_len))
    
    count_arr1[0][np.arange(read1_pos[0], read1_pos[1])] += 1
    count_arr2[0][np.arange(read2_pos[0], read2_pos[1])] += 1
    
    read1_arr = np.array(read1.query_alignment_sequence[read1_pos[0]:read1_pos[1]])
    read2_arr = np.array(read2.query_alignment_sequence[read2_pos[0]:read2_pos[1]])
    nonmatch_idx = np.squeeze(np.where(read1_arr != read2_arr)) # identify the mismatch region
    
    # identify which read is correct, which read is not
    for idx in nonmatch_idx:
        read1_base = read1_arr[idx]
        read2_base = read2_arr[idx]
        ref_pos    = 1
        ref_base   = ref_dict[contig_id][ref_pos: ref_pos].seq
        
        read2_seq = read2.query_alignment_sequence[(s1-s2):(s1-s2:end-start)]
        ref_seq = ref_dict[read1.reference_name][start:end].seq
        strand= read1.get_tag('YS').split("_")[0]
        
    
    if read1.get_tag('YS') == "W_C2T" and read1.aligned_pairs[0] == read2.aligned_pairs[0]:
        break

In [32]:
for read1, read2 in read_pair_generator(bam_file, "chr21"):
    if read1.get_tag('YS') == "W_C2T" and read1.aligned_pairs[0] == read2.aligned_pairs[0]:
        break

In [33]:
read1.seq

'GGGAGTGGTGGGGGTGTTGGGGGTTGAGTGTGGAGGAGTTGGGAGTGGTGGGGGAGTTGGGGGTTGATTGTGGAGGAGTTGGGAGTGGTGGGGGAGTTGGGGGTTGAGTGTGGAGGAG'

In [34]:
read1.get_tag('YS')

'W_C2T'

In [35]:
read1.cigarstring

'118M'

In [36]:
read2.cigarstring

'118M'

In [37]:
read1.reference_start

5114234

In [38]:
read1.reference_end

5114352

In [39]:
read2.seq

'GGGGGGGGGGGGGGTGTGGGGGGGGGCGGGGGGCGGAGTTGGGTGTGGGGGGGGTGTTGGGGGTTGATTGTGGAGGAGTTGGGAGTGGTGGGGGAGTTGGGGGTTGAGTGTGGAGGAG'

In [ ]:
read2.get_tag('YS')

In [ ]:
read2.reference_start

In [ ]:
read2.reference_end

In [ ]:
str(ref_dict['chr21'][read2.reference_start:read2.reference_end].seq)

In [ ]:
for ix in range(len(read2.seq)):
    if read2.seq[ix] == read1.seq[ix]:
        continue
    else:
        print(ix)

In [ ]:
read1.seq[86]

In [ ]:
 read2.seq[82]

In [ ]:
read1.query_alignment_qualities

In [ ]:
read1.query_qualities

In [ ]:
read1.qual

In [ ]:
for read1, read2 in gen_read_pair(bam_file, "chr21"):
    if read1.reference_start < read2.reference_start and read2.reference_end > read1.reference_end:
        if read2.reference_start - read1.reference_end <= 0:
            break
    

In [ ]:
[read1.reference_start, read1.reference_end, read2.reference_start, read2.reference_end]

In [ ]:
overlap_len = read2.reference_start - read1.reference_end

In [ ]:
overlap_len

In [ ]:
read1.seq

In [ ]:
read2.seq

In [ ]:
read1.seq[overlap_len:]

In [ ]:
read2.seq[:  ]

In [ ]:
read2.query_sequence

In [ ]:
read2.tags

In [ ]:
read1.tags

In [ ]:
read1.flag

In [ ]:
read1.infer_read_length()

In [ ]:
read2.infer_read_length()

In [ ]:
for read1, read2 in gen_read_pair(bam_file, "chr21"):
    overlap_len = read2.reference_start - read1.reference_end
    if overlap_len <= 0:
        read1_overlap = read1.seq[-overlap_len:]
        read2_overlap = read2.seq[:overlap_len]
        
    
        
        
    
    print(read1)
    break

In [ ]:
read2.positions

In [ ]:
read1.aligned_pairs

In [ ]:
read2.aligned_pairs

In [ ]:
pos_arr

In [ ]:
read1.seq

In [ ]:
read2.seq

In [ ]:
# get read1 and read2
# get the overlapped region
# compare the overlapped region



In [ ]:
read1.get_overlap(read2)

In [ ]:
read1.aligned_pairs

In [ ]:
read2.aligned_pairs

In [ ]:
read1.get_overlap()